**CAMADA BRONZE - INGESTÃO E ARMAZENAMENTO DOS DADOS BRUTOS**

In [0]:
#LEITURA E INSPECIONAMENTO DOS ARQUIOS EM XLSX


In [0]:
# Lendo o Excel da Telemetria 
df_telemetria_excel = spark.read.format("excel") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/estudo-guiado/vale/dados_brutos/datasets/datasets/telemetria/desenvolver_dontgo.xlsx")

# Mostrando se ele conseguiu ler e a quantidade de linhas
print(f"Total de linhas encontradas no Excel de Telemetria: {df_telemetria_excel.count()}")
display(df_telemetria_excel.limit(5))

Total de linhas encontradas no Excel de Telemetria: 148


_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10,_c11,_c12,_c13,_c14,_c15,_c16,_c17,_c18
Id_Eventos_Telemetria,Data_Evento,Inicio_Turno,Fim_Turno,Dia,Localidade,TAG,Tag_Frota,Porte,Tipo,Nome_Operador_Anon,Matricula_Operador_Hash,Id_Alarme,Alarme,Id_Criticidade,Criticidade,Valor,Classe,Is_Dont_Go
9371424,2025-01-01 00:05:01,2024-12-31 18:00:00,2025-01-01 06:00:00,1,Itabira,CA65924,793-D 4S,Grande Porte,Caminhao,OP_067,H_ce3d6b9eae65,335609875,2nd Tire Tag Timeout,3,Informacional,0,null,0
9371423,2025-01-01 00:05:01,2024-12-31 18:00:00,2025-01-01 06:00:00,1,Itabira,CA65924,793-D 4S,Grande Porte,Caminhao,OP_067,H_ce3d6b9eae65,335609862,1st Tire Tag Timeout,3,Informacional,0,null,0
9371430,2025-01-01 00:05:05,2024-12-31 18:00:00,2025-01-01 06:00:00,1,Itabira,CA65924,793-D 4S,Grande Porte,Caminhao,OP_067,H_ce3d6b9eae65,335609935,Rx Channel B Not Receiving Messages,3,Informacional,0,null,0
9371429,2025-01-01 00:05:05,2024-12-31 18:00:00,2025-01-01 06:00:00,1,Itabira,CA65924,793-D 4S,Grande Porte,Caminhao,OP_067,H_ce3d6b9eae65,335609934,Rx Channel A Not Receiving Messages,3,Informacional,0,null,0


In [0]:
# Lendo o Excel de Apontamentos 
df_apontamentos_excel = spark.read.format("excel") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/estudo-guiado/vale/dados_brutos/datasets/datasets/apontamentos/desenvolver_apontamentos.xlsx")

# Mostrando o resultado na tela
print(f"Total de linhas encontradas no Excel de Apontamentos: {df_apontamentos_excel.count()}")
display(df_apontamentos_excel.limit(5))

Total de linhas encontradas no Excel de Apontamentos: 101


_c0,_c1,_c2,_c3,_c4,_c5,_c6
Id,Inicio,Fim,Tag,Frota,Tipo,Classe
23462554,2025-01-04 19:12:54,2025-01-04 19:40:29,CA65789,793-D 2S,Caminhao,Operando
23462555,2025-01-04 19:12:28,2025-01-04 19:38:15,CA65908,793-D 3S,Caminhao,Operando
23462556,2025-01-04 19:14:07,2025-01-04 19:18:29,CA65915,793-D 4S,Caminhao,Parado
23462759,2025-01-04 19:20:36,2025-01-04 19:27:44,CA5926,793-D 5S,Caminhao,Parado


agora que verificamos os arquivos em xlsx vamos ver os arquivos em parquet

In [0]:
# Lendo os arquivos Parquet da pasta de Telemetria
df_telemetria_parquet = spark.read.format("parquet").load("/Volumes/estudo-guiado/vale/dados_brutos/datasets/datasets/telemetria/*.parquet")

print(f"Total de linhas no Parquet da Telemetria: {df_telemetria_parquet.count()}")

Total de linhas no Parquet da Telemetria: 37164054


In [0]:
# ==============================================================================
# ESTRATÉGIA DEFINITIVA: PONTE PANDAS -> SPARK (À PROVA DE FALHAS)
# ==============================================================================

import pandas as pd

# 1. Caminho oficial do arquivo único
caminho_apontamento = "//Volumes/estudo-guiado/vale/dados_brutos/datasets/datasets/apontamentos/desenvolver_apontamentos.parquet"

# 2. Lemos usando o Pandas (ele aceita os nanossegundos nativamente sem reclamar!)
pdf_apontamentos = pd.read_parquet(caminho_apontamento)

# 3. Convertemos as colunas de data para string no Pandas para o Spark não chiar
pdf_apontamentos['Inicio'] = pdf_apontamentos['Inicio'].astype(str)
pdf_apontamentos['Fim'] = pdf_apontamentos['Fim'].astype(str)

# 4. Transformamos o DataFrame do Pandas em um DataFrame do Spark
df_apontamentos_parquet = spark.createDataFrame(pdf_apontamentos)

# 5. Mostra o total de linhas real e os dados na tela
print(f"Total de linhas em Apontamentos: {df_apontamentos_parquet.count()}")
display(df_apontamentos_parquet.limit(5))

Total de linhas em Apontamentos: 377907


Id,Inicio,Fim,Tag,Frota,Tipo,Classe
23462554,2025-01-04 19:12:54,2025-01-04 19:40:29,CA65789,793-D 2S,Caminhao,Operando
23462555,2025-01-04 19:12:28,2025-01-04 19:38:15,CA65908,793-D 3S,Caminhao,Operando
23462556,2025-01-04 19:14:07,2025-01-04 19:18:29,CA65915,793-D 4S,Caminhao,Parado
23462759,2025-01-04 19:20:36,2025-01-04 19:27:44,CA5926,793-D 5S,Caminhao,Parado
23462762,2025-01-04 19:18:29,2025-01-04 19:23:51,CA65915,793-D 4S,Caminhao,Operando


In [0]:
# Lendo o Excel de Alarmes (Regra de Negócio)
df_alarmes_excel = spark.read.format("excel") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/estudo-guiado/vale/dados_brutos/datasets/datasets/Alarmes - Regra de Negocio.xlsx")

print(f"Total de linhas no catálogo de regras: {df_alarmes_excel.count()}")
display(df_alarmes_excel.limit(5))

Total de linhas no catálogo de regras: 152


_c0,_c1,_c2,_c3,_c4,_c5
TIPO,EVENTO,SITUACAO,QTD,TEMPO,NIVEL
ALARME OEM,Low Transmission Oil Level,Mediante alarme nível 3,2,360,Muito Alto
ALARME OEM,Low Transmission Oil Level,Mediante cinco alarmes nivel 2 consecutivos.,5,360,Muito alto
ALARME OEM,Transmission Oil Level - Active,Mediante alarme nível 3,2,360,Muito Alto
ALARME OEM,Engine Coolant Level - Active,Mediante alarme nível 3,1,0,Muito Alto



Após a validação inicial dos dados brutos na pasta de `apontamentos`, identificamos que o arquivo Excel (`desenvolver_apontamentos.xlsx`) possui exatamente a mesma estrutura e conteúdo de negócio que o arquivo Parquet (`desenvolver_apontamentos.parquet`). 

**Decisão Estratégica:**
Optamos por seguir o pipeline oficial utilizando **exclusivamente os arquivos no formato Parquet** pelos seguintes motivos:
1. **Volumetria Real:** Os arquivos Parquet contêm a carga completa e o histórico de produção (como visto nos múltiplos arquivos mensais da Telemetria), enquanto o Excel é apenas uma amostra reduzida.
2. **Performance:** O formato Parquet é colunar e otimizado para processamento distribuído no Apache Spark, garantindo maior eficiência e menor consumo de memória na leitura da camada Bronze para a Silver.
3. **Tratamento de Tipos:** O erro nativo de interpretação de nanossegundos (`TIMESTAMP(NANOS)`) do arquivo de apontamentos foi mitigado com sucesso utilizando a conversão via Pandas e isolando o arquivo Parquet alvo, eliminando a necessidade de leitura da planilha Excel intrusa.

In [0]:
# ==============================================================================
#GRAVAÇÃO CONSOLIDADA NA CAMADA VALE_BRONZE

# Definimos os caminhos exatos apontando para o banco de dados 'vale_bronze'
tabela_apontamentos = "`estudo-guiado`.vale_bronze.apontamentos"
tabela_telemetria = "`estudo-guiado`.vale_bronze.telemetria"
tabela_alarmes = "`estudo-guiado`.vale_bronze.alarmes_regras"

# 1. Salva os Apontamentos como Tabela Delta no banco vale_bronze
df_apontamentos_parquet.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabela_apontamentos)

# 2. Salva a Telemetria (todos os meses juntos!) como Tabela Delta no banco vale_bronze
df_telemetria_parquet.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabela_telemetria)

#3. Salvar Alarmes como Tabela Delta no banco vale_bronze
df_alarmes_excel.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabela_alarmes)

print("======================================================================")
print(" Camada Bronze gravada com sucesso!")
print(f" -> {tabela_apontamentos}")
print(f" -> {tabela_telemetria}")
print(f" -> {tabela_alarmes}")
print("======================================================================")

 Camada Bronze gravada com sucesso!
 -> `estudo-guiado`.vale_bronze.apontamentos
 -> `estudo-guiado`.vale_bronze.telemetria
 -> `estudo-guiado`.vale_bronze.alarmes_regras
